# Scaling-Clamp Sweep (BACKLOG §1.8)

**Date:** 2026-04-18.
**Question:** Does loosening `_compute_scaling`'s upper clamp (currently 2.0) help the high-volume under-prediction at T-3d without hurting cohort-wide MAE?

**Context from F-audit (findings §17):** phase 1 KDE systematically under-predicts high-volume h/m movies at T-3d (mean_err −14.78 on 5 targets). Scaling is hitting the 2.0 upper cap and still under-shooting. Findings §9.3 swept threshold + lower clamp but kept upper clamp at 2.0 throughout — this is the gap.

**Test design:**
- Cohort-wide LOO at T-3d, T-5d, T-7d.
- Ship stack: `combined_score(α=0.5, σ_gap=8) + ceil=0.7d + n=20`.
- Sweep upper clamp ∈ {2.0, 3.0, 4.0, ∞}. Lower clamp stays 0.5, threshold stays 40.
- Measure `phase 1 MAE vs actual_phase1` (not full-window), isolating the scaling mechanism from the phase 2 form.
- Stratify by `actual_phase1` quartile (direct volume signal) and `close_day_count` quartile (shipping-usable proxy).
- Decision rule: non-worse on cohort AND meaningfully better on high-volume subset → ship looser clamp.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from _helpers import (
    reviews, close_date_map, gaps, gap_lookup, first_review_ts,
    gap_for_slug,
    combined_score_selector,
    snapshot_state, actual_remaining, close_day_count,
    build_critic_profiles, build_kde_lambda_model_capped,
    predict_window_custom,
    passes_skip_rules_for_snap,
    CACHE_DIR,
)

SHIP_ALPHA = 0.5
SHIP_SIGMA_GAP = 8.0
SHIP_N_TRAINING = 20
SHIP_BANDWIDTH_FLOOR = 0.5
SHIP_BANDWIDTH_CEIL = 0.7
SCALING_THRESHOLD = 40.0
CLAMP_LOWER = 0.5

SNAPS = [3.0, 5.0, 7.0]
UPPER_CLAMPS = [2.0, 3.0, 4.0, float('inf')]

CLAMP_CACHE_PATH = CACHE_DIR / 'scaling_clamp_sweep.pkl'

print(f'Cohort: {len(close_date_map)} resolved movies')
print(f'Snaps: {SNAPS}')
print(f'Upper clamps to sweep: {UPPER_CLAMPS}')
print(f'Cache: {CLAMP_CACHE_PATH.name}')

## LOO sweep

For each (target, snap), build profiles+KDE once, then predict phase 1 with each upper_clamp variant. Ground truth `actual_phase1` = reviews with `midnight_utc_dbc < dbc <= snap_dbc` (excludes pre-market close-day, which is phase 2's territory).

In [ ]:
def run_clamp_sweep(force=False, verbose=True):
    if CLAMP_CACHE_PATH.exists() and not force:
        cached = pd.read_pickle(CLAMP_CACHE_PATH)
        print(f'Loaded {len(cached)} cached rows')
        return cached

    rows = []
    for snap in SNAPS:
        if verbose:
            print(f'--- T-{snap:g}d ---')
        for i, target in enumerate(close_date_map):
            target_gap = gap_for_slug(target)
            if target_gap is None:
                continue
            target_close = close_date_map[target]
            midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
            snap_time = target_close - pd.Timedelta(days=snap)
            state = snapshot_state(target, snap_time)
            passed, reason = passes_skip_rules_for_snap(state, snap)
            if not passed:
                continue

            target_window_days = state['first_review_dbc'] - snap
            target_critics = state['observed_critics']

            training, _ = combined_score_selector(
                target, target_gap, target_critics, target_window_days,
                k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
            )
            if len(training) < 5:
                continue

            profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
            model = build_kde_lambda_model_capped(
                profiles,
                bandwidth_floor=SHIP_BANDWIDTH_FLOOR,
                bandwidth_ceiling=SHIP_BANDWIDTH_CEIL,
            )

            # Ground truth phase 1: reviews strictly before midnight UTC of close day
            movie_reviews = reviews[reviews['movie_slug'] == target].copy()
            movie_reviews['dbc'] = (target_close - movie_reviews['estimated_timestamp']).dt.total_seconds() / 86400
            actual_p1 = int(((movie_reviews['dbc'] > midnight_utc_dbc) & (movie_reviews['dbc'] <= snap)).sum())

            # Training movie's close-day count (shipping-usable volume proxy)
            cd_target = close_day_count(target)

            # Predict phase 1 for each upper_clamp variant
            row = {
                'target_slug': target, 'snap_dbc': snap,
                'actual_phase1': actual_p1,
                'cd_target': cd_target,
                'observed_count': state['observed_count'],
                'n_critics_observed': len(target_critics),
                'first_review_dbc': state['first_review_dbc'],
            }
            for uc in UPPER_CLAMPS:
                pred = predict_window_custom(
                    model, dbc_from=snap, dbc_to=midnight_utc_dbc,
                    observed_critics=target_critics,
                    observed_count=state['observed_count'],
                    first_review_dbc=state['first_review_dbc'],
                    scaling_threshold=SCALING_THRESHOLD,
                    scaling_clamp=(CLAMP_LOWER, uc),
                )
                uc_label = 'inf' if np.isinf(uc) else f'{uc:g}'
                row[f'pred_uc={uc_label}'] = float(pred)
            rows.append(row)
            if verbose and (i + 1) % 30 == 0:
                print(f'  {i+1}/{len(close_date_map)} targets')

    df = pd.DataFrame(rows)
    df.to_pickle(CLAMP_CACHE_PATH)
    print(f'Cached {len(df)} rows')
    return df

results = run_clamp_sweep()
print()
print('Rows per snap:')
print(results.groupby('snap_dbc').size().to_string())

## Cohort-wide MAE by clamp and snap

In [ ]:
uc_labels = ['2', '3', '4', 'inf']
uc_cols = [f'pred_uc={lbl}' for lbl in uc_labels]

cohort_summary = []
for snap in SNAPS:
    sub = results[results['snap_dbc'] == snap].copy()
    if not len(sub):
        continue
    for lbl, col in zip(uc_labels, uc_cols):
        err = sub[col] - sub['actual_phase1']
        cohort_summary.append({
            'snap': f'T-{snap:g}d',
            'upper_clamp': lbl,
            'n': len(sub),
            'MAE': err.abs().mean(),
            'mean_err': err.mean(),
            'median_err': err.median(),
        })
cohort_df = pd.DataFrame(cohort_summary)
print('Cohort-wide phase-1 MAE vs actual_phase1 by (snap, upper_clamp):')
print(cohort_df.to_string(index=False, float_format='%.3f'))

## Stratified by actual_phase1 quartile

High-volume stratum = Q4 of actual_phase1 within each snap. Shows whether looser clamp helps the movies that motivated this test.

In [ ]:
strat_summary = []
for snap in SNAPS:
    sub = results[results['snap_dbc'] == snap].copy()
    if len(sub) < 4:
        continue
    sub['q_actual'] = pd.qcut(sub['actual_phase1'], q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')
    for q, qsub in sub.groupby('q_actual'):
        for lbl, col in zip(uc_labels, uc_cols):
            err = qsub[col] - qsub['actual_phase1']
            strat_summary.append({
                'snap': f'T-{snap:g}d',
                'actual_q': str(q),
                'n': len(qsub),
                'actual_p1_range': f'{qsub["actual_phase1"].min()}-{qsub["actual_phase1"].max()}',
                'upper_clamp': lbl,
                'MAE': err.abs().mean(),
                'mean_err': err.mean(),
            })
strat_df = pd.DataFrame(strat_summary)
print('MAE stratified by actual_phase1 quartile (high = Q4):')
for snap in SNAPS:
    snap_label = f'T-{snap:g}d'
    snap_rows = strat_df[strat_df['snap'] == snap_label]
    if not len(snap_rows):
        continue
    print(f'\n=== {snap_label} ===')
    pivot = snap_rows.pivot_table(
        index=['actual_q', 'n', 'actual_p1_range'],
        columns='upper_clamp', values='MAE',
    )
    pivot = pivot[uc_labels]
    print(pivot.to_string(float_format='%.2f'))

## Stratified by close_day_count quartile (shipping-usable volume proxy)

A movie's `close_day_count` is observable at deployment time (it's a lagging measure from review history). Stratifying by it mirrors the shippability question: "does looser clamp help for the kinds of movies we can identify as likely high-volume?" 

In [ ]:
strat2_summary = []
for snap in SNAPS:
    sub = results[results['snap_dbc'] == snap].copy()
    if len(sub) < 4:
        continue
    sub['q_cd'] = pd.qcut(sub['cd_target'], q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')
    for q, qsub in sub.groupby('q_cd'):
        for lbl, col in zip(uc_labels, uc_cols):
            err = qsub[col] - qsub['actual_phase1']
            strat2_summary.append({
                'snap': f'T-{snap:g}d',
                'cd_q': str(q),
                'n': len(qsub),
                'cd_range': f'{qsub["cd_target"].min()}-{qsub["cd_target"].max()}',
                'upper_clamp': lbl,
                'MAE': err.abs().mean(),
                'mean_err': err.mean(),
            })
strat2_df = pd.DataFrame(strat2_summary)
print('MAE stratified by close_day_count quartile:')
for snap in SNAPS:
    snap_label = f'T-{snap:g}d'
    snap_rows = strat2_df[strat2_df['snap'] == snap_label]
    if not len(snap_rows):
        continue
    print(f'\n=== {snap_label} ===')
    pivot = snap_rows.pivot_table(
        index=['cd_q', 'n', 'cd_range'],
        columns='upper_clamp', values='MAE',
    )
    pivot = pivot[uc_labels]
    print(pivot.to_string(float_format='%.2f'))

## Mean-error stratification (diagnosis)

Shows the sign of the error — is each stratum over- or under-predicting? Helps confirm the "Q4 under, Q1-Q2 over" pattern and whether looser clamp closes it.

In [ ]:
print('MEAN_ERR by actual_phase1 quartile (negative = under-prediction):')
for snap in SNAPS:
    snap_label = f'T-{snap:g}d'
    snap_rows = strat_df[strat_df['snap'] == snap_label]
    if not len(snap_rows):
        continue
    print(f'\n=== {snap_label} ===')
    pivot = snap_rows.pivot_table(
        index=['actual_q', 'n'],
        columns='upper_clamp', values='mean_err',
    )
    pivot = pivot[uc_labels]
    print(pivot.to_string(float_format='%+.2f'))

## H/m-subset sanity check

Re-verify the F-audit finding on these 5 movies specifically — does looser clamp help them the way the diagnosis predicted?

In [ ]:
HM_TARGETS = [
    'the_drama', 'the_super_mario_galaxy_movie',
    'forbidden_fruits_2026', 'they_will_kill_you', 'you_me_and_tuscany',
]
hm = results[results['target_slug'].isin(HM_TARGETS)].copy()
print(f'H/m subset: {len(hm)} (target, snap) rows')
print()
for snap in SNAPS:
    sub = hm[hm['snap_dbc'] == snap].copy()
    if not len(sub):
        continue
    print(f'=== T-{snap:g}d ===')
    cols = ['target_slug', 'actual_phase1'] + uc_cols
    print(sub[cols].to_string(index=False, float_format='%.2f'))
    print()
    for lbl, col in zip(uc_labels, uc_cols):
        err = sub[col] - sub['actual_phase1']
        print(f'  clamp={lbl:3s}  MAE={err.abs().mean():6.2f}  mean_err={err.mean():+7.2f}')
    print()

## Decision

Read the tables above:

1. **Cohort-wide MAE at T-3d:** does any looser clamp match or beat clamp=2.0?
2. **Q4 actual_phase1 subset:** does looser clamp reduce MAE / mean_err magnitude?
3. **Q1-Q2 strata:** does looser clamp hurt the low-volume movies (by letting scaling over-correct upward on noise)?
4. **H/m sanity:** do `they_will_kill_you` and `the_drama` move the right direction?

If the answer is "clamp_hi=X wins on high-vol without hurting low-vol at T-3d," ship it — update BACKLOG §1.5 to specify the clamp tuple as a parameter, and the library integration lands with the new default.

If loosening hurts too much on low-vol: logged as a dead-end mitigation; the issue moves to Path B (volume feature in `combined_score`).